In [ ]:
%pip install -e "/Workspace/Users/josorioos@argos.com.co/genie_assessment_library"

In [ ]:
import json
import os
import subprocess
from pathlib import Path
from typing import Any

OUTPUT_FILE_PATTERNS = (
    "genie_evidence_payload_*.json",
    "genie_final_client_report_*.json",
    "genie_scorecard_*.xlsx",
    "genie_proposed_metric_view_*.yml",
    "genie_proposed_metric_view_*.yaml",
)

dbutils: Any = globals().get("dbutils")
if dbutils is None:
    raise RuntimeError("Este notebook debe ejecutarse en Databricks (dbutils no disponible).")


def clear_previous_assessment_outputs() -> None:
    """Elimina salidas previas del assessment para evitar mezclar corridas."""
    candidate_directories = [
        Path.cwd(),
        Path.cwd() / "assessment_outputs",
        Path.cwd() / "genie_assessment" / "temp" / "assessment_outputs",
    ]
    for directory in candidate_directories:
        if not directory.exists():
            continue
        for pattern in OUTPUT_FILE_PATTERNS:
            for file_path in directory.glob(pattern):
                if file_path.is_file():
                    file_path.unlink()


def read_config(config_workspace_path):
    """Lee el config desde Workspace Files o desde el directorio local del notebook."""
    candidates = [
        Path(config_workspace_path),
        Path.cwd() / "genie_assessment" / "temp" / "config.json",
        Path.cwd() / "assessment_config.json",
    ]

    for candidate in candidates:
        try:
            if candidate.is_file():
                return candidate.read_text(encoding="utf-8"), candidate
        except OSError:
            continue

    try:
        config_content = dbutils.fs.head(f"file:{config_workspace_path}")
        return config_content, Path("assessment_config.json")
    except Exception as error:
        searched_paths = ", ".join(str(candidate) for candidate in candidates)
        raise FileNotFoundError(
            f"No se encontró el config. Rutas revisadas: {searched_paths}. "
            f"Ruta DBFS/Workspace: {config_workspace_path}. Error: {error}"
        ) from error


dbutils.widgets.text(
    "config_workspace_path",
    "/Workspace/Users/${workspace.current_user.userName}/.bundle/template_databricks_asset_bundle/dev/files/genie_assessment/temp/config.json",
    "Config JSON path",
)

config_workspace_path = dbutils.widgets.get("config_workspace_path")
config_content, config_source = read_config(config_workspace_path)
local_config_path = Path("assessment_config.json")

if config_source != local_config_path:
    local_config_path.write_text(config_content, encoding="utf-8")

with local_config_path.open(encoding="utf-8") as config_file:
    config = json.load(config_file)

if not config.get("warehouse_id"):
    raise ValueError("El config no contiene warehouse_id")

clear_previous_assessment_outputs()

context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
os.environ["DATABRICKS_HOST"] = context.apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = context.apiToken().get()

result = subprocess.run(
    ["genie-assess", "--config", str(local_config_path)],
    capture_output=True,
    text=True,
    env=os.environ.copy(),
)

print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"genie-assess terminó con código {result.returncode}")